## NumPy & Real File Formats (Parquet, JSON)

### 3.1 — Why NumPy in DE?
Pandas is built on top of NumPy. You won't use NumPy directly every day, but you need it for:

- Fast mathematical operations on large arrays
- Understanding how Pandas works under the hood
- Working with ML libraries (PyTorch, scikit-learn all speak NumPy)

In [1]:
import numpy as np

# --- Arrays (like lists but faster and math-friendly)
arr = np.array([1, 2, 3, 4, 5])
print(arr * 2)        # [2 4 6 8 10] — operates on all elements at once
print(arr + 10)       # [11 12 13 14 15]
print(arr.mean())     # 3.0
print(arr.std())      # standard deviation
print(arr.sum())      # 15

# --- 2D arrays (like a table)
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
print(matrix.shape)   # (3, 3)
print(matrix[0])      # first row: [1 2 3]
print(matrix[:, 1])   # second column: [2 5 8]

# --- Useful for DE: handling nulls mathematically
scores = np.array([72, 68, np.nan, 61, 65])
print(np.nanmean(scores))   # mean ignoring NaN — 66.5
print(np.nanmax(scores))    # max ignoring NaN

[ 2  4  6  8 10]
[11 12 13 14 15]
3.0
1.4142135623730951
15
(3, 3)
[1 2 3]
[2 5 8]
66.5
72.0


### 3.2 — NumPy + Pandas Together

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "district": ["Gasabo", "Huye", "Musanze"],
    "population": [600000, 330000, 420000],
    "health_score": [72, 61, np.nan]
})

# np.where — vectorized if/else (faster than .apply() for simple conditions)
df["is_large"] = np.where(df["population"] > 400000, True, False)

# np.log — useful for normalizing skewed data
df["pop_log"] = np.log(df["population"])

# Replace nulls using numpy
df["health_score"] = df["health_score"].fillna(np.nanmean(df["health_score"]))

# Percentile — useful for outlier detection
p95 = np.percentile(df["population"].dropna(), 95)
print(f"95th percentile population: {p95}")

95th percentile population: 582000.0


When to use `np.where` vs `.apply()`:

Simple condition (if/else on one column) → `np.where` — faster
Complex logic (multiple conditions, custom function) → `.apply()`

### 3.3 — Parquet Files: The DE Standard Format

In Data Engineering, Parquet is the most important file format after CSV. Here's why:
| Format  | Size  | Speed | Best for                 |
| ------- | ----- | ----- | ------------------------ |
| CSV     | Large | Slow  | Sharing, Excel           |
| JSON    | Large | Slow  | APIs, configs            |
| Parquet | Small | Fast  | Pipelines, cloud storage |

Parquet is columnar — it stores data by column, not by row. This means if you only need 2 columns from a 100-column file, it only reads those 2. Massive speed gain on large datasets.

In [3]:
# Install: pip install pyarrow
import pandas as pd

df = pd.DataFrame({
    "district": ["Gasabo", "Huye", "Musanze"],
    "population": [600000, 330000, 420000],
    "health_score": [72.0, 61.0, 65.0]
})

# Save to Parquet
df.to_parquet("districts.parquet", index=False)

# Read from Parquet
df_loaded = pd.read_parquet("districts.parquet")
print(df_loaded)

# Read only specific columns — this is the big advantage
df_partial = pd.read_parquet("districts.parquet", columns=["district", "population"])
print(df_partial)

  district  population  health_score
0   Gasabo      600000          72.0
1     Huye      330000          61.0
2  Musanze      420000          65.0
  district  population
0   Gasabo      600000
1     Huye      330000
2  Musanze      420000


### 3.4 — JSON Files: For APIs and Configs

You'll constantly deal with JSON when calling APIs:

In [5]:
import json
import pandas as pd

# =========================
# 1. CREATE SAMPLE JSON FILE
# =========================
data = [
    {
        "name": "Gasabo",
        "province": "Kigali City",
        "location": {
            "lat": -1.94,
            "lon": 30.06
        },
        "stats": {
            "population": 600000,
            "health_score": 72
        }
    },
    {
        "name": "Kicukiro",
        "province": "Kigali City",
        "location": {
            "lat": -1.97,
            "lon": 30.10
        },
        "stats": {
            "population": 400000,
            "health_score": 68
        }
    },
    {
        "name": "Musanze",
        "province": "Northern",
        "location": {
            "lat": -1.50,
            "lon": 29.63
        },
        "stats": {
            "population": 420000,
            "health_score": 75
        }
    }
]

# Write JSON file
with open("data.json", "w") as f:
    json.dump(data, f, indent=4)


# =========================
# 2. READ JSON FILE
# =========================
with open("data.json", "r") as f:
    raw_data = json.load(f)

print("Raw JSON loaded successfully!")
print(type(raw_data))


# =========================
# 3. NORMAL JSON → DATAFRAME
# =========================
df = pd.DataFrame(raw_data)
print("\nBasic DataFrame:")
print(df.head())


# =========================
# 4. FLATTEN NESTED JSON
# =========================
df_flat = pd.json_normalize(raw_data, sep="_")

print("\nFlattened DataFrame Columns:")
print(df_flat.columns.tolist())


# =========================
# 5. BASIC CLEANING + TRANSFORMATIONS
# =========================

# Convert population to millions
df_flat["stats_population_millions"] = df_flat["stats_population"] / 1_000_000

# Categorize population size
def categorize(pop):
    if pop > 500000:
        return "Large"
    elif pop > 300000:
        return "Medium"
    else:
        return "Small"

df_flat["size_category"] = df_flat["stats_population"].apply(categorize)

# Rename columns for clarity
df_flat = df_flat.rename(columns={
    "name": "district",
    "stats_population": "population",
    "stats_health_score": "health_score"
})


# =========================
# 6. GROUP BY ANALYSIS
# =========================
summary = df_flat.groupby("province").agg(
    total_population=("population", "sum"),
    avg_health=("health_score", "mean"),
    district_count=("district", "count")
).reset_index()

print("\nSummary by province:")
print(summary)


# =========================
# 7. FINAL OUTPUT
# =========================
print("\nFinal Cleaned Data:")
print(df_flat)

Raw JSON loaded successfully!
<class 'list'>

Basic DataFrame:
       name     province                      location  \
0    Gasabo  Kigali City  {'lat': -1.94, 'lon': 30.06}   
1  Kicukiro  Kigali City   {'lat': -1.97, 'lon': 30.1}   
2   Musanze     Northern   {'lat': -1.5, 'lon': 29.63}   

                                        stats  
0  {'population': 600000, 'health_score': 72}  
1  {'population': 400000, 'health_score': 68}  
2  {'population': 420000, 'health_score': 75}  

Flattened DataFrame Columns:
['name', 'province', 'location_lat', 'location_lon', 'stats_population', 'stats_health_score']

Summary by province:
      province  total_population  avg_health  district_count
0  Kigali City           1000000        70.0               2
1     Northern            420000        75.0               1

Final Cleaned Data:
   district     province  location_lat  location_lon  population  \
0    Gasabo  Kigali City         -1.94         30.06      600000   
1  Kicukiro  Kigali City 

### 3.5 — Comparing File Sizes

In [7]:
import os

# Create a larger dataset to see the difference
import numpy as np
df_large = pd.DataFrame({
    "district": ["Gasabo"] * 100000,
    "population": np.random.randint(100000, 700000, 100000),
    "health_score": np.random.uniform(50, 90, 100000)
})

df_large.to_csv("large.csv", index=False)
df_large.to_parquet("large.parquet", index=False)

csv_size = os.path.getsize("large.csv")
parquet_size = os.path.getsize("large.parquet")

print(f"CSV:     {csv_size / 1024:.1f} KB")
print(f"Parquet: {parquet_size / 1024:.1f} KB")
print(f"Parquet is {csv_size / parquet_size:.1f}x smaller")
# Typical result: Parquet is 5-10x smaller

CSV:     3219.4 KB
Parquet: 1559.0 KB
Parquet is 2.1x smaller


## Exercises 

Exercise 1 — Create a NumPy array of 10 population values. Calculate the mean, max, min, and standard deviation. Then create a second array of health scores (include one `np.nan`) and calculate the mean ignoring the NaN.

In [8]:
import numpy as np

# Population array
populations = np.array([600000, 400000, 350000, 330000, 420000, 390000, 500000, 450000, 380000, 410000])

print("Mean:", populations.mean())
print("Max:", populations.max())
print("Min:", populations.min())
print("Std Dev:", populations.std())

# Health scores with NaN
health_scores = np.array([72, 68, np.nan, 61, 65, 70, 75, 80, np.nan, 66])

print("Mean (ignoring NaN):", np.nanmean(health_scores))

Mean: 423000.0
Max: 600000
Min: 330000
Std Dev: 74572.11275000863
Mean (ignoring NaN): 69.625


Exercise 2 — Using the districts dataset from Lesson 2, add two new columns using `np.where`:

- `high_poverty`: `True` if `poverty_pct` > 25, else `False`
- `pop_category`: `"High"` if population > 450000, `"Medium"` if > 300000, else `"Low"` — use `np.select` for this one (look it up — it handles multiple conditions)

In [9]:
import pandas as pd
import numpy as np

# Recreate cleaned dataset
data = [
    {"district": "Gasabo",     "province": "Kigali City", "population": 600000, "health_score": 72, "poverty_pct": 15.2},
    {"district": "Kicukiro",   "province": "Kigali City", "population": 400000, "health_score": 68, "poverty_pct": 18.5},
    {"district": "Nyarugenge", "province": "Kigali City", "population": 350000, "health_score": 66.0, "poverty_pct": 20.1},
    {"district": "Huye",       "province": "Southern",    "population": 330000, "health_score": 61, "poverty_pct": 35.0},
    {"district": "Musanze",    "province": "Northern",    "population": 420000, "health_score": 65, "poverty_pct": 28.3},
    {"district": "Rubavu",     "province": "Western",     "population": 390000, "health_score": 63, "poverty_pct": 30.1},
]

df = pd.DataFrame(data)

# --- high_poverty using np.where
df["high_poverty"] = np.where(df["poverty_pct"] > 25, True, False)

# --- pop_category using np.select
conditions = [
    df["population"] > 450000,
    df["population"] > 300000
]

choices = ["High", "Medium"]

df["pop_category"] = np.select(conditions, choices, default="Low")

print(df)

     district     province  population  health_score  poverty_pct  \
0      Gasabo  Kigali City      600000          72.0         15.2   
1    Kicukiro  Kigali City      400000          68.0         18.5   
2  Nyarugenge  Kigali City      350000          66.0         20.1   
3        Huye     Southern      330000          61.0         35.0   
4     Musanze     Northern      420000          65.0         28.3   
5      Rubavu      Western      390000          63.0         30.1   

   high_poverty pop_category  
0         False         High  
1         False       Medium  
2         False       Medium  
3          True       Medium  
4          True       Medium  
5          True       Medium  


Exercise 3 — Take the cleaned DataFrame from Lesson 2, save it as Parquet, then read it back. Then read it again but only load the `district` and `health_score` columns. Print the file sizes of the CSV vs Parquet versions.

In [10]:
import pandas as pd
import numpy as np
import os

# Sample cleaned DataFrame
df = pd.DataFrame({
    "district": ["Gasabo", "Huye", "Musanze"],
    "population": [600000, 330000, 420000],
    "health_score": [72.0, 61.0, 65.0]
})

# Save files
df.to_csv("districts.csv", index=False)
df.to_parquet("districts.parquet", index=False)

# Read full parquet
df_parquet = pd.read_parquet("districts.parquet")
print(df_parquet)

# Read only selected columns
df_partial = pd.read_parquet("districts.parquet", columns=["district", "health_score"])
print(df_partial)

# Compare sizes
csv_size = os.path.getsize("districts.csv")
parquet_size = os.path.getsize("districts.parquet")

print(f"CSV size: {csv_size} bytes")
print(f"Parquet size: {parquet_size} bytes")
print(f"Parquet is {csv_size / parquet_size:.2f}x smaller")

  district  population  health_score
0   Gasabo      600000          72.0
1     Huye      330000          61.0
2  Musanze      420000          65.0
  district  health_score
0   Gasabo          72.0
1     Huye          61.0
2  Musanze          65.0
CSV size: 93 bytes
Parquet size: 2366 bytes
Parquet is 0.04x smaller


Exercise 4 — Parse this nested JSON and flatten it into a clean DataFrame using `pd.json_normalize()`:

In [11]:
import pandas as pd

api_response = [
    {"id": 1, "district": "Gasabo",  "metrics": {"health": 72, "poverty": 15.2}, "location": {"province": "Kigali City"}},
    {"id": 2, "district": "Huye",    "metrics": {"health": 61, "poverty": 35.0}, "location": {"province": "Southern"}},
    {"id": 3, "district": "Musanze", "metrics": {"health": 65, "poverty": 28.3}, "location": {"province": "Northern"}},
]

# Flatten
df = pd.json_normalize(api_response, sep="_")

# Rename columns
df = df.rename(columns={
    "metrics_health": "health_score",
    "metrics_poverty": "poverty_pct",
    "location_province": "province"
})

print(df)
print(df.columns.tolist())

   id district  health_score  poverty_pct     province
0   1   Gasabo            72         15.2  Kigali City
1   2     Huye            61         35.0     Southern
2   3  Musanze            65         28.3     Northern
['id', 'district', 'health_score', 'poverty_pct', 'province']
